# 에너지 어보브 헐 실습

**Energy Above Hull · E_hull · 볼록껍질**

같은 조성의 가장 안정한 상 조합보다 얼마나 에너지가 높은지 나타내는 값.

소재 분야에서 이해하기: 0에 가까운 후보를 우선 검토하되 합성 가능성과 동일시하지 않는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 볼록껍질을 직접 만들어 봅니다

이원계에서 조성별 형성 에너지의 하부 볼록껍질을 구하고, 각 후보가 그보다 얼마나 위에 있는지 계산합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# x = B 분율, y = 형성 에너지 (eV/atom). 양 끝(순수 원소)은 0 입니다.
entries = [
    ('A',      0.00,  0.00),
    ('A3B',    0.25, -0.32),
    ('AB',     0.50, -0.55),
    ('AB2',    0.67, -0.40),
    ('AB3',    0.75, -0.28),
    ('B',      1.00,  0.00),
    ('A2B*',   0.33, -0.30),
    ('AB(hp)', 0.50, -0.41),
    ('AB4*',   0.80, -0.10),
]
names = [entry[0] for entry in entries]
x = np.array([entry[1] for entry in entries])
energy = np.array([entry[2] for entry in entries])
print('후보 %d개' % len(entries))

In [ ]:
def lower_hull(x, energy):
    """하부 볼록껍질 위의 점 인덱스를 x 순서로 반환."""
    order = np.argsort(x)
    hull = []
    for index in order:
        while len(hull) >= 2:
            (x1, y1), (x2, y2) = (x[hull[-2]], energy[hull[-2]]), (x[hull[-1]], energy[hull[-1]])
            cross = (x2 - x1) * (energy[index] - y1) - (x[index] - x1) * (y2 - y1)
            if cross <= 0:      # 마지막 점이 껍질 위가 아니면 버립니다
                hull.pop()
            else:
                break
        hull.append(index)
    return hull

hull = lower_hull(x, energy)
print('껍질 위의 상:', [names[index] for index in hull])
hull_energy = np.interp(x, x[hull], energy[hull])
above = energy - hull_energy
for name, value, distance in zip(names, energy, above):
    flag = '안정(껍질 위)' if distance < 1e-9 else '준안정'
    print('%-8s E_form %+.3f   E_above_hull %.3f  %s' % (name, value, distance, flag))

In [ ]:
grid = np.linspace(0, 1, 200)
plt.plot(grid, np.interp(grid, x[hull], energy[hull]), 'k-', label='convex hull')
plt.scatter(x, energy, s=40, c=['green' if value < 1e-9 else 'orange' for value in above])
for name, xi, yi in zip(names, x, energy):
    plt.annotate(name, (xi, yi), textcoords='offset points', xytext=(4, 6), fontsize=8)
plt.axhline(0, color='gray', lw=0.8)
plt.xlabel('B fraction'); plt.ylabel('formation energy (eV/atom)'); plt.legend(); plt.show()
print('주황색 점은 형성 에너지가 음수여도 껍질 위 상들의 조합으로 분해되는 것이 더 유리합니다.')
print('E_above_hull 0.05 eV/atom 이하를 합성 가능 후보로 보는 관행이 있지만, 합성 가능성과 같은 뜻은 아닙니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#energy-above-hull)을 여세요.